### Gold Products Pipeline

This notebook defines the Gold layer SCD Type 2 product dimension in Lakeflow Spark Declarative Pipelines.

* Source table: `databricks_cata.silver.product_silver`
* Target dataset: `databricks_cata.gold.dimproducts`
* Execution mode: serverless pipeline refresh
* Pattern used: snapshot-based Auto CDC with SCD Type 2 history

In [0]:
from pyspark import pipelines as dp

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-8567633090960622>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import *

ModuleNotFoundError: No module named 'dlt'

### Dataset Design

This notebook uses snapshot-based Auto CDC to build an SCD Type 2 Gold dimension.

* `dimproducts` is created as the target table in the pipeline target schema `databricks_cata.gold`.
* The source snapshot is `databricks_cata.silver.product_silver`.
* No column projection is hardcoded, so new source columns can flow through automatically.
* The pipeline manages historical versions with `__START_AT` and `__END_AT`.

In [0]:
dp.create_streaming_table(
    name="dimproducts",
    comment="Gold SCD Type 2 product dimension built from databricks_cata.silver.product_silver snapshots."
)

dp.create_auto_cdc_from_snapshot_flow(
    target="dimproducts",
    source="databricks_cata.silver.product_silver",
    keys=["product_id"],
    stored_as_scd_type=2,
)

### Publishing Notes

After a successful refresh, this notebook creates and loads `databricks_cata.gold.dimproducts`.

The table is maintained as SCD Type 2 history, so current records have `__END_AT IS NULL` and historical versions are preserved automatically.